In [ ]:
import numpy as np
import pandas as pd
import json
import config

In [ ]:
# Check lims dsn connection to database before proceeding
def get_lims_connection():
    global odbcLimsConnection
    if config.dbType == "oracle":
        import oracledb
        odbcLimsConnection = oracledb.connect(user=config.limsdb_username, password=config.limsdb_password,dsn=config.limsdb_dsn)
        
    elif config.dbType == "postgres":
        import psycopg2
        odbcLimsConnection = psycopg2.connect('dbname=' + config.limsdb_database + ' user=' + config.limsdb_username + 
                                             ' password=' + config.limsdb_password + ' host=' + config.limsdb_hostname)
    
    else:
        import pyodbc
        odbcLimsConnection = pyodbc.connect('DRIVER=' + config.limsdb_driver + ';DATABASE=' + config.limsdb_database +
                                        ';UID=' + config.limsdb_username + ';PWD=' + config.limsdb_password +
                                        ';SERVER=' + config.limsdb_hostname + ';PORT=' + config.limsdb_port +
                                        ';Trusted_Connection=yes')
    return odbcLimsConnection


if __name__ == '__main__':
    try:  # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_lims_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)


# Check mlar dsn connection to database before proceeding
def get_mlar_connection():
    global odbcMlarConnection
    if config.dbType == "oracle":
        import oracledb
        odbcMlarConnection = oracledb.connect(user=config.mlardb_username, password=config.mlardb_password,dsn=config.mlardb_dsn)
    elif config.dbType == "postgres":
        import psycopg2
        odbcMlarConnection = psycopg2.connect('dbname=' + config.mlardb_database + ' user=' + config.mlardb_username + 
                                             ' password=' + config.mlardb_password + ' host=' + config.mlardb_hostname)        
    else:
        import pyodbc
        odbcMlarConnection = pyodbc.connect('DRIVER=' + config.mlardb_driver + ';DATABASE=' + config.mlardb_database +
                                        ';UID=' + config.mlardb_username + ';PWD=' + config.mlardb_password +
                                        ';SERVER=' + config.mlardb_hostname + ';PORT=' + config.mlardb_port +
                                        ';Trusted_Connection=yes')
    return odbcMlarConnection


if __name__ == '__main__':
    try:
        # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_mlar_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)


In [ ]:
column_names = ['sample_number','audit_timestamp','audit_type','trans_string']
sql_command = "select distinct SAMPLE_NUMBER, AUDIT_TIMESTAMP, AUDIT_TYPE, TRANS_STRING from SAMPLE_AUDIT_DECRYPT"
dataFrame = pd.read_sql(sql_command, odbcMlarConnection, columns=column_names)


In [ ]:
bkupDataFrame = dataFrame
#dataFrame = bkupDataFrame
#bkupDataFrame

In [ ]:
#JSON data transformation
dataFrame['trans_string'] = dataFrame.trans_string.str.replace("\r\n", " ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace("\n", " ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace("\r", " ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'" ', r"> ", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r' "', r" <", regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r' <,"', r' <","' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'> <"', r'" "' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'<}', r'"}' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'""[A-Z]', r'"[A-Z]' , regex=True)
dataFrame['trans_string'] = dataFrame.trans_string.str.replace(r'\\', r' ' , regex=True)

In [ ]:
def loads_json(x):
   try:
       return json.loads(x, strict=False) if pd.notna(x) else pd.Series() #
   except json.JSONDecodeError as e:
       print(f"Error decoding JSON: {e}")
       print(f"Problematic JSON string: {x}")
       return pd.Series()
    
dataFrame['trans_string'] = dataFrame['trans_string'].apply(loads_json)
jsonDataFrame = pd.json_normalize(dataFrame['trans_string'])

In [ ]:
#dataFrame

In [ ]:
dataFrame = dataFrame.drop(columns=['trans_string'])
jsonDataFrame = jsonDataFrame.drop(columns=['SAMPLE_NUMBER']).replace("▄", ",", regex=True) #.replace(np.nan, "█")
encodedDataFrame = pd.concat([dataFrame, jsonDataFrame], axis=1)

In [ ]:
standard_labware_fields = ['TEXT_ID','STATUS','OLD_STATUS','IN_SPEC','IN_CAL','RE_SAMPLE','ORIGINAL_SAMPLE','ALIQUOT','PARENT_SAMPLE','PARENT_ALIQUOT','LOGIN_DATE','LOGIN_BY','RECD_DATE','RECEIVED_BY','DATE_STARTED','STARTED','DATE_COMPLETED','DATE_REVIEWED','PREP_DATE','PREP_BY','PRODUCT_VERSION','PROJECT','BATCH_NAME','BATCH_TEMPLATE','STEP','SAMPLE_TYPE','TEMPLATE','STANDARD','LOT','CHANGED_ON','PARTIAL_SPEC','PRIMARY_IN_SPEC','BATCH','PROTOCOL_LEVEL','RELEASED','RELEASED_BY','RELEASED_ON','IN_CONTROL','INVESTIGATION','INVESTIGATED','MODIFIED_RESULTS','COMPOSITE','PARENT_COMPOSITE','HAS_FLAGS','IMPORTANT_MESSAGE','SIGNED','TEST_NUMBER','CHARGE_ENTRY','ANALYSIS','VERSION','REPLICATE_COUNT','STATUS','OLD_STATUS','DATE_RECEIVED','DATE_STARTED','PREP','PREP_DATE','PREP_BY','DATE_COMPLETED','DATE_REVIEWED','REVIEWER','REPLICATE_TEST','IN_SPEC','IN_CAL','ORDER_NUMBER','GROUP_NAME','RESOLVE_REQD','CHANGED_ON','STAGE','PRIMARY_IN_SPEC','BATCH','IN_CONTROL','VARIATION','RE_TESTED','TEST_LIST','MODIFIED_RESULTS','ALIQUOTED_TO','ON_WORKSHEET','ANALYSIS_COUNT','BATCH_ORIGINAL_TEST','BATCH_PARENT_TEST','CHILD_OUT_SPEC','CROSS_SAMPLE','ORIGINAL_TEST','PARENT_TEST','RELEASED','RELEASED_BY','RELEASED_ON','SIGNED','RESULT_NUMBER','TEST_NUMBER','NAME','REPLICATE_COUNT','ORDER_NUMBER','RESULT_TYPE','MINIMUM','MAXIMUM','ALLOW_OUT','FORMATTED_ENTRY','NUMERIC_ENTRY','PLACES','STATUS','OLD_STATUS','ENTERED_ON','ENTERED_BY','USES_CODES','AUTO_CALC','LIST_KEY','ALLOW_CANCEL','IN_SPEC','REPORTABLE','OPTIONAL','CODE_ENTERED','ROUND','DATE_REVIEWED','REVIEWER','CHANGED_ON','STD_REAG_SAMPLE','DISPLAYED','ENTRY_QUALIFIER','INSTRUMENT','ANALYSIS','USES_INSTRUMENT','IN_CAL','HAS_ATTRIBUTES','FACTOR_OPERATOR','ENTRY_TYPE','ALIAS_NAME','CONTROL_1','CONTROL_2','IN_CONTROL','REPORTED_NAME','PRIMARY_IN_SPEC']
audit_fields = ['audit_timestamp','audit_type','sample_number']
filter_fields = standard_labware_fields + audit_fields 
encodedDataFrame = encodedDataFrame.filter(items=filter_fields)

In [ ]:
encodedDataFrame